# 📊 Agregatsiya va Guruhlash (PostgreSQL)

## 🎯 Mavzu: SQL aggregation funksiyalari va GROUP BY

### 📋 Reja:
1. Agregatsiya funksiyalari (COUNT, SUM, AVG, MIN, MAX)
2. GROUP BY operatori
3. HAVING filterlash
4. Murakkab agregatsiya so'rovlari

---

## 🔌 PostgreSQL ga ulanish

Birinchi navbatda kerakli kutubxonalarni import qilamiz va bazaga ulanamiz:

In [ ]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns

# PostgreSQL ulanish parametrlari
connection_params = {
    'host': 'localhost',
    'database': 'lesson_aggregation',
    'user': 'data_scientist',
    'password': 'secure_password123',
    'port': '5432'
}

# SQLAlchemy engine yaratish
engine = create_engine(f"postgresql://{connection_params['user']}:{connection_params['password']}@"
                      f"{connection_params['host']}:{connection_params['port']}/{connection_params['database']}")

print("✅ PostgreSQL ga muvaffaqiyatli ulanildi!")
print("\n📊 Dars boshlashga tayyormiz!")

## 1️⃣ Agregatsiya Funksiyalari

### 📝 Asosiy funksiyalar:

1. **COUNT()** - yozuvlar sonini hisoblash
2. **SUM()** - yig'indini hisoblash
3. **AVG()** - o'rtacha qiymatni hisoblash
4. **MIN()** - minimal qiymatni topish
5. **MAX()** - maksimal qiymatni topish

### 🔍 COUNT() funksiyasi

In [ ]:
# Jami xodimlar soni
query = """
SELECT COUNT(*) as total_employees
FROM employees;
"""

df = pd.read_sql(query, engine)
print("👥 Jami xodimlar:", df['total_employees'][0])

# Bo'limlar bo'yicha xodimlar soni
query = """
SELECT 
    d.department_name,
    COUNT(e.employee_id) as employee_count
FROM departments d
LEFT JOIN employees e ON d.department_id = e.department_id
GROUP BY d.department_name
ORDER BY employee_count DESC;
"""

df = pd.read_sql(query, engine)
print("\n📊 Bo'limlar bo'yicha xodimlar:")
print(df.to_string(index=False))

### 💰 SUM() funksiyasi

In [ ]:
# Umumiy oylik maosh fondi
query = """
SELECT 
    SUM(salary) as total_salary,
    SUM(salary)/COUNT(*) as avg_salary,
    MIN(salary) as min_salary,
    MAX(salary) as max_salary
FROM employees;
"""

df = pd.read_sql(query, engine)
print("💰 Oylik maosh statistikasi:")
print(f"Jami fond: {df['total_salary'][0]:,.0f} so'm")
print(f"O'rtacha: {df['avg_salary'][0]:,.0f} so'm")
print(f"Minimal: {df['min_salary'][0]:,.0f} so'm")
print(f"Maksimal: {df['max_salary'][0]:,.0f} so'm")

# Bo'limlar bo'yicha maosh fondi
query = """
SELECT 
    d.department_name,
    COUNT(e.employee_id) as employee_count,
    SUM(e.salary) as total_salary,
    AVG(e.salary) as avg_salary
FROM departments d
LEFT JOIN employees e ON d.department_id = e.department_id
GROUP BY d.department_name
ORDER BY total_salary DESC;
"""

df = pd.read_sql(query, engine)
print("\n📊 Bo'limlar bo'yicha maosh fondi:")
print(df.to_string(index=False))

### 📊 AVG() funksiyasi

O'rtacha qiymatlarni hisoblash:

In [ ]:
# Mahsulotlar statistikasi
query = """
SELECT 
    c.category_name,
    COUNT(p.product_id) as products_count,
    AVG(p.price) as avg_price,
    AVG(p.rating) as avg_rating,
    AVG(p.stock_quantity) as avg_stock
FROM categories c
LEFT JOIN products p ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY avg_price DESC;
"""

df = pd.read_sql(query, engine)
print("🛍️ Kategoriyalar bo'yicha mahsulotlar statistikasi:")
print(df.to_string(index=False))

# Vizualizatsiya
plt.figure(figsize=(12, 6))
plt.bar(df['category_name'], df['avg_price'])
plt.xticks(rotation=45)
plt.title('Kategoriyalar bo\'yicha o\'rtacha narx')
plt.xlabel('Kategoriya')
plt.ylabel("O'rtacha narx (so'm)")
plt.tight_layout()
plt.show()

### 📈 MIN() va MAX() funksiyalari

Minimal va maksimal qiymatlarni topish:

In [ ]:
# Har bir bo'limdagi yosh statistikasi
query = """
SELECT 
    d.department_name,
    MIN(e.age) as min_age,
    MAX(e.age) as max_age,
    AVG(e.age) as avg_age,
    COUNT(e.employee_id) as employee_count
FROM departments d
LEFT JOIN employees e ON d.department_id = e.department_id
GROUP BY d.department_name
ORDER BY avg_age DESC;
"""

df = pd.read_sql(query, engine)
print("👥 Bo'limlar bo'yicha yosh statistikasi:")
print(df.to_string(index=False))

# Box plot vizualizatsiyasi
query = """
SELECT 
    d.department_name,
    e.age
FROM departments d
LEFT JOIN employees e ON d.department_id = e.department_id
WHERE e.age IS NOT NULL;
"""

df = pd.read_sql(query, engine)

plt.figure(figsize=(12, 6))
sns.boxplot(x='department_name', y='age', data=df)
plt.xticks(rotation=45)
plt.title('Bo\'limlar bo\'yicha yosh taqsimoti')
plt.xlabel('Bo\'lim')
plt.ylabel('Yosh')
plt.tight_layout()
plt.show()

## 2️⃣ GROUP BY Operatori

### 📝 GROUP BY ning vazifasi:
- Ma'lumotlarni guruhlash
- Har bir guruh uchun agregat funksiyalarni qo'llash
- Natijalarni guruhlar bo'yicha ko'rish

### 🔍 Guruhlash misollarini ko'rib chiqamiz:

In [ ]:
# Mijozlar statistikasi (hudud bo'yicha)
query = """
SELECT 
    region,
    COUNT(*) as customer_count,
    COUNT(CASE WHEN is_active = true THEN 1 END) as active_customers,
    AVG(total_spent) as avg_spent,
    MAX(total_spent) as max_spent
FROM customers
GROUP BY region
ORDER BY customer_count DESC;
"""

df = pd.read_sql(query, engine)
print("👥 Hududlar bo'yicha mijozlar statistikasi:")
print(df.to_string(index=False))

# Vizualizatsiya
plt.figure(figsize=(12, 6))
plt.bar(df['region'], df['customer_count'], label='Jami mijozlar')
plt.bar(df['region'], df['active_customers'], label='Faol mijozlar')
plt.xticks(rotation=45)
plt.title('Hududlar bo\'yicha mijozlar')
plt.xlabel('Hudud')
plt.ylabel('Mijozlar soni')
plt.legend()
plt.tight_layout()
plt.show()

### 📈 Ko'p ustun bo'yicha guruhlash:

In [ ]:
# Sotuvlar statistikasi (hudud va kategoriya bo'yicha)
query = """
SELECT 
    o.delivery_region as region,
    c.category_name,
    COUNT(DISTINCT s.sale_id) as sales_count,
    SUM(s.total_price) as total_revenue,
    AVG(s.total_price) as avg_sale,
    SUM(s.profit_margin) as total_profit
FROM sales s
JOIN products p ON s.product_id = p.product_id
JOIN categories c ON p.category_id = c.category_id
JOIN orders o ON s.order_id = o.order_id
GROUP BY o.delivery_region, c.category_name
ORDER BY region, total_revenue DESC;
"""

df = pd.read_sql(query, engine)
print("📊 Hudud va kategoriya bo'yicha sotuvlar:")
print(df.to_string(index=False))

# Pivot table ko'rinishida vizualizatsiya
pivot_df = df.pivot(index='category_name', columns='region', values='total_revenue')
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_df, annot=True, fmt='.0f', cmap='YlOrRd')
plt.title('Hudud va kategoriya bo\'yicha sotuvlar (heatmap)')
plt.tight_layout()
plt.show()

## 3️⃣ HAVING Operatori

### 🎯 HAVING ning vazifasi:
- GROUP BY natijalarini filtrlash
- Agregat funksiyalar natijasiga shart qo'yish
- WHERE dan farqi - guruhlangan ma'lumotlar bilan ishlaydi

### 📝 HAVING misollarini ko'ramiz:

In [ ]:
# Yuqori savdoli hududlar
query = """
SELECT 
    o.delivery_region as region,
    COUNT(DISTINCT o.order_id) as orders_count,
    COUNT(DISTINCT o.customer_id) as unique_customers,
    SUM(o.total_amount) as total_revenue
FROM orders o
GROUP BY o.delivery_region
HAVING SUM(o.total_amount) > 1000000000  -- 1 milliard so'mdan yuqori
ORDER BY total_revenue DESC;
"""

df = pd.read_sql(query, engine)
print("💰 Yuqori savdoli hududlar (1 mlrd so'mdan yuqori):")
print(df.to_string(index=False))

# Vizualizatsiya
plt.figure(figsize=(10, 6))
plt.scatter(df['orders_count'], df['total_revenue'], 
           s=df['unique_customers']*10, alpha=0.6)

for i, row in df.iterrows():
    plt.annotate(row['region'], 
                 (row['orders_count'], row['total_revenue']))

plt.title('Yuqori savdoli hududlar')
plt.xlabel('Buyurtmalar soni')
plt.ylabel('Jami savdo (so\'m)')
plt.tight_layout()
plt.show()

In [ ]:
# Faol sotuvchilar bo'limlar bo'yicha
query = """
SELECT 
    d.department_name,
    COUNT(DISTINCT e.employee_id) as employees_count,
    COUNT(DISTINCT o.order_id) as orders_count,
    SUM(o.total_amount) as total_sales,
    SUM(o.total_amount)/COUNT(DISTINCT e.employee_id) as sales_per_employee
FROM departments d
JOIN employees e ON d.department_id = e.department_id
JOIN orders o ON e.employee_id = o.employee_id
GROUP BY d.department_name
HAVING COUNT(DISTINCT o.order_id) >= 100  -- 100+ buyurtma qabul qilgan bo'limlar
ORDER BY sales_per_employee DESC;
"""

df = pd.read_sql(query, engine)
print("👥 Eng faol savdo bo'limlari (100+ buyurtma):")
print(df.to_string(index=False))

# Bar chart vizualizatsiyasi
plt.figure(figsize=(10, 6))
x = range(len(df['department_name']))
width = 0.35

plt.bar(x, df['total_sales']/1e6, width, label='Jami savdo (mln)')
plt.bar([i+width for i in x], df['sales_per_employee']/1e6, 
        width, label='Xodim boshiga (mln)')

plt.xticks([i+width/2 for i in x], df['department_name'], rotation=45)
plt.title('Bo\'limlar bo\'yicha savdo ko\'rsatkichlari')
plt.ylabel('Million so\'m')
plt.legend()
plt.tight_layout()
plt.show()

## 4️⃣ Murakkab Agregatsiya So'rovlari

### 🎯 Murakkab so'rovlar:
- Bir nechta jadvallarni JOIN qilish
- Ko'p ustunli GROUP BY
- Shartli agregatsiya (CASE bilan)
- Sub-querylar

### 📊 Misollar:

In [ ]:
# Mahsulot kategoriyalari va mijoz turlari bo'yicha sotuvlar
query = """
WITH sales_data AS (
    SELECT 
        c.category_name,
        cust.customer_type,
        COUNT(DISTINCT s.sale_id) as sales_count,
        SUM(s.total_price) as total_revenue,
        SUM(s.profit_margin) as total_profit,
        AVG(s.total_price) as avg_sale
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    JOIN categories c ON p.category_id = c.category_id
    JOIN orders o ON s.order_id = o.order_id
    JOIN customers cust ON o.customer_id = cust.customer_id
    GROUP BY c.category_name, cust.customer_type
)
SELECT 
    category_name,
    customer_type,
    sales_count,
    total_revenue,
    ROUND(total_profit/total_revenue * 100, 2) as profit_margin_percent,
    avg_sale
FROM sales_data
ORDER BY total_revenue DESC;
"""

df = pd.read_sql(query, engine)
print("📊 Kategoriya va mijoz turi bo'yicha sotuvlar:")
print(df.to_string(index=False))

# Vizualizatsiya
pivot_df = df.pivot(index='category_name', 
                    columns='customer_type', 
                    values='total_revenue')

plt.figure(figsize=(12, 8))
sns.heatmap(pivot_df/1e6, annot=True, fmt='.1f', cmap='YlOrRd')
plt.title('Kategoriya va mijoz turi bo\'yicha sotuvlar (mln so\'m)')
plt.tight_layout()
plt.show()

In [ ]:
# Xodimlar, bo'limlar va oylar bo'yicha sotuvlar dinamikasi
query = """
WITH monthly_sales AS (
    SELECT 
        d.department_name,
        TO_CHAR(o.order_date, 'YYYY-MM') as month,
        COUNT(DISTINCT o.order_id) as orders_count,
        COUNT(DISTINCT o.customer_id) as customers_count,
        SUM(o.total_amount) as total_sales,
        COUNT(DISTINCT e.employee_id) as active_employees
    FROM departments d
    JOIN employees e ON d.department_id = e.department_id
    JOIN orders o ON e.employee_id = o.employee_id
    GROUP BY d.department_name, TO_CHAR(o.order_date, 'YYYY-MM')
)
SELECT 
    department_name,
    month,
    orders_count,
    customers_count,
    total_sales,
    ROUND(total_sales/active_employees, 2) as sales_per_employee
FROM monthly_sales
ORDER BY department_name, month;
"""

df = pd.read_sql(query, engine)
print("📈 Oylik sotuvlar dinamikasi:")
print(df.to_string(index=False))

# Line chart vizualizatsiyasi
plt.figure(figsize=(12, 6))

for dept in df['department_name'].unique():
    dept_data = df[df['department_name'] == dept]
    plt.plot(dept_data['month'], 
             dept_data['total_sales']/1e6, 
             marker='o', 
             label=dept)

plt.title('Bo\'limlar bo\'yicha oylik sotuvlar dinamikasi')
plt.xlabel('Oy')
plt.ylabel('Sotuvlar (mln so\'m)')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

## 📝 Xulosa

PostgreSQL da agregatsiya va guruhlash operatorlari juda kuchli va ko'p qirrali. Ular yordamida:

1. **Statistik tahlil** - COUNT, SUM, AVG va boshqalar
2. **Guruhlash** - GROUP BY bilan ma'lumotlarni kategoriyalash
3. **Filtrlash** - HAVING bilan guruhlangan natijalarni filtrlash
4. **Murakkab tahlillar** - JOIN, CTE, Sub-query va boshqalar

### 🎯 Keyingi qadamlar:
- `practical.ipynb` da amaliy mashqlarni bajaring
- `group_practice.ipynb` da guruhda ishlang
- `homework.ipynb` da uy vazifalarini bajaring

---

*🚀 Endi amaliy mashg'ulotlarni boshlashingiz mumkin!*